# Macro Customization Tutorial

This notebook covers **four workflows** for customizing macros on QUAM quantum-dot machines:

1. **Use defaults** — Wire built-in macros with no overrides.
2. **Type-level override** — Replace a macro for all instances of a component type (e.g. all `QuantumDot`).
3. **Instance-level override** — Replace a macro for one specific component (e.g. `quantum_dots.virtual_dot_1`).
4. **External macro package** — Import overrides from a separate Python package.

**Runs without QM hardware** — no `qm.open()`, `qm.run()`, or `machine.connect()` required. All code builds QUA programs in memory only.

## Workflow 1 — Use Defaults

Wire the machine with default macros only. No profile, no overrides.

In [1]:
from quam_builder.architecture.quantum_dots.examples.tutorial_machine import build_tutorial_machine
from quam_builder.architecture.quantum_dots.macro_engine import wire_machine_macros
from qm import qua

machine = build_tutorial_machine()
wire_machine_macros(machine)

# Verify quantum_dots, quantum_dot_pairs, sensor_dots have expected macros
qd = next(iter(machine.quantum_dots.values()))
pair = next(iter(machine.quantum_dot_pairs.values()))
sd = next(iter(machine.sensor_dots.values()))
assert "initialize" in qd.macros and "empty" in qd.macros
assert "initialize" in pair.macros and "measure" in pair.macros and "empty" in pair.macros
assert "measure" in sd.macros
print("QuantumDot:", sorted(qd.macros.keys()))
print("QuantumDotPair:", sorted(pair.macros.keys()))
print("SensorDot:", sorted(sd.macros.keys()))

# Build a small QUA program (no run)
q1, q2 = machine.qubits["q1"], machine.qubits["q2"]
with qua.program() as prog:
    q1.initialize()
    q2.initialize()
    pair.initialize()
    sd.measure('readout')
    q1.measure()
    q2.measure()
print("Built QUA program successfully (Workflow 1).")

2026-04-30 18:34:15,983 - qm - INFO     - Starting session: a19726fe-7b35-4eef-84ed-91136d8e1957
QuantumDot: ['align', 'empty', 'initialize', 'wait']
QuantumDotPair: ['align', 'empty', 'initialize', <VoltagePointName.MEASURE: 'measure'>, 'wait']
SensorDot: ['align', <VoltagePointName.MEASURE: 'measure'>, 'wait']
Built QUA program successfully (Workflow 1).


/Users/sebastian/Documents/GitHub/superconducting_qualibrate/CS_installations/.venv/lib/python3.12/site-packages/quam/core/quam_classes.py:291: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: ReadoutResonatorSingle
  warnings.warn(
/Users/sebastian/Documents/GitHub/superconducting_qualibrate/CS_installations/.venv/lib/python3.12/site-packages/quam/core/quam_classes.py:291: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component: VirtualGateSet
  warnings.warn(
/Users/sebastian/Documents/GitHub/superconducting_qualibrate/CS_installations/.venv/lib/python3.12/site-packages/quam/core/quam_classes.py:291: UserWarning: This component is not part of any QuamRoot, using last instantiated QuamRoot. This is not recommended as it may lead to unexpected behaviour. Component

## Workflow 2 — Type-Level Override

Override the `initialize` macro for **all** `QuantumDot` components with a custom class.

In [2]:
from functools import partial

from quam.core import quam_dataclass
from quam.core.macro import QuamMacro
from quam_builder.architecture.quantum_dots.operations.default_macros.state_macros import InitializeStateMacro
from quam_builder.architecture.quantum_dots.operations.names import VoltagePointName
from quam_builder.architecture.quantum_dots.operations.macro_catalog import TypeOverrideCatalog
from quam_builder.architecture.quantum_dots.components import QuantumDot


@quam_dataclass
class CustomInitMacro(InitializeStateMacro):
    """Custom initialize macro with different ramp_duration."""
    ramp_duration: int = 64


machine = build_tutorial_machine()
wire_machine_macros(
    machine,
    catalogs=[
        TypeOverrideCatalog({
            QuantumDot: {
                VoltagePointName.INITIALIZE: partial(CustomInitMacro, ramp_duration=64),
            },
        }),
    ],
)

# Verify override applied
qd = next(iter(machine.quantum_dots.values()))
init_macro = qd.macros["initialize"]
assert isinstance(init_macro, CustomInitMacro)
assert init_macro.ramp_duration == 64
print("QuantumDot initialize is CustomInitMacro with ramp_duration=", init_macro.ramp_duration)

QuantumDot initialize is CustomInitMacro with ramp_duration= 64


## Workflow 3 — Instance-Level Override

Override a macro for **one** specific component using its path (`quantum_dots.<id>`, etc.).

In [3]:
@quam_dataclass
class MyInitMacro(InitializeStateMacro):
    """Instance-specific custom initialize macro."""
    ramp_duration: int = 96


machine = build_tutorial_machine()
wire_machine_macros(
    machine,
    instance_overrides={
        "quantum_dots.virtual_dot_1": {
            VoltagePointName.INITIALIZE: MyInitMacro,
        },
    },
)

# Verify only virtual_dot_1 has override
qd1 = machine.quantum_dots["virtual_dot_1"]
qd2 = machine.quantum_dots["virtual_dot_2"]
assert isinstance(qd1.macros["initialize"], MyInitMacro)
assert not isinstance(qd2.macros["initialize"], MyInitMacro)
print("virtual_dot_1.initialize:", type(qd1.macros["initialize"]).__name__)
print("virtual_dot_2.initialize:", type(qd2.macros["initialize"]).__name__)

virtual_dot_1.initialize: MyInitMacro
virtual_dot_2.initialize: InitializeStateMacro


## Workflow 4 — External Macro Package

In practice, your lab keeps custom macros in a separate package. Here we simulate that with an inline `build_macro_overrides()` function.

In [4]:
class LabMacroCatalog:
    """Simulates an external package catalog implementing MacroCatalog protocol.

    In practice this lives in my_lab_qd_macros.catalog.
    """
    priority = 200

    def get_factories(self, component_type):
        if issubclass(component_type, QuantumDot):
            return {
                VoltagePointName.INITIALIZE: partial(CustomInitMacro, ramp_duration=64),
            }
        return {}


machine = build_tutorial_machine()
wire_machine_macros(machine, catalogs=[LabMacroCatalog()])

qd = next(iter(machine.quantum_dots.values()))
assert isinstance(qd.macros["initialize"], CustomInitMacro)
print("External catalog applied: QuantumDot.initialize = CustomInitMacro")

External catalog applied: QuantumDot.initialize = CustomInitMacro


## @quam_dataclass Anti-Pattern

Custom macro classes **must** use `@quam_dataclass` and be defined in a proper Python module for save/load round-trips. Notebook-defined classes (even with the decorator) have no module path and fail on load. The demo shows BadMacro (notebook-defined) failing vs InitializeStateMacro (from the library) surviving.

In [5]:
import tempfile
from pathlib import Path
from quam_builder.architecture.quantum_dots.operations.default_macros.state_macros import (
    InitializeStateMacro,
)


# Bad: no @quam_dataclass, defined in notebook — no module path for deserialization
class BadMacro(QuamMacro):
    def __init__(self, ramp_duration=16):
        self.ramp_duration = ramp_duration

    def apply(self, **kwargs):
        pass


machine = build_tutorial_machine()
wire_machine_macros(machine)

# Assign BadMacro, save, load — load fails (notebook-defined class has no module path)
qd = next(iter(machine.quantum_dots.values()))
qd.macros["initialize"] = BadMacro(ramp_duration=32)
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "machine.json"
    machine.save(path)
    try:
        loaded = type(machine).load(path)
        assert False, "Load should have failed"
    except ValueError as e:
        print("BadMacro (no decorator, notebook-defined): load failed as expected")

# Good: @quam_dataclass from a proper module — survives save/load
machine2 = build_tutorial_machine()
wire_machine_macros(machine2)
qd2 = next(iter(machine2.quantum_dots.values()))
qd2.macros["initialize"] = InitializeStateMacro(ramp_duration=48)
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "machine.json"
    machine2.save(path)
    loaded2 = type(machine2).load(path)
ld_qd2 = next(iter(loaded2.quantum_dots.values()))
assert isinstance(ld_qd2.macros["initialize"], InitializeStateMacro)
assert ld_qd2.macros["initialize"].ramp_duration == 48
print("InitializeStateMacro (@quam_dataclass from library) survived save/load with ramp_duration=48")


BadMacro (no decorator, notebook-defined): load failed as expected
InitializeStateMacro (@quam_dataclass from library) survived save/load with ramp_duration=48


/Users/sebastian/Documents/GitHub/superconducting_qualibrate/CS_installations/.venv/lib/python3.12/site-packages/quam/utils/general.py:39: UserWarning: Could not determine the module of BadMacro, this may cause issues when trying to load QUAM from a file. Please ensure that all QUAM classes are defined in a Python module
  warnings.warn(
